## Dataset Preparation: Image Downloading and Validation

This notebook prepares the multimodal dataset by downloading images, validating cached files, resizing them to a consistent resolution, and exporting cleaned CSV files for later training notebooks.

### Import the Core Utilities and Mount Google Drive

This cell imports the libraries used for downloading images, reading metadata, and handling image files, then mounts Google Drive so the dataset can be prepared in persistent Colab storage.

In [ ]:
from urllib import request
from PIL import Image
import os
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

### Load the Initial Metadata Table

This cell loads the initial cleaned metadata file and removes columns that are not needed for the multimodal preparation step.

In [ ]:
df = pd.read_csv(
    '/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/multimodal_train.tsv',
    sep='\t'
)
df.drop(['2_way_label', '3_way_label', 'title'], axis = 1, inplace =True)
df.head(10)

### Define the Main Output Paths

This cell defines the directory used to cache downloaded images and the path used to store the cleaned metadata table.

In [ ]:
IMAGEDIR = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images"
CLEANDFPATH = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv"


### Create a Working Subset and Inspect the Data

This cell creates a smaller working subset with stratified sampling and performs basic checks on null values and class balance before downloading begins.

In [ ]:

from sklearn.model_selection import train_test_split

df, df_backup = train_test_split(
    df,
    test_size=0.95,
    shuffle=True,
    # To maintain percentage of samples per class as given by original dataset
    stratify=df["6_way_label"]
)
df.reset_index(drop=True, inplace=True)
df
print(df['clean_title'].isnull().sum())
print(df['id'].isnull().sum())
print(df['hasImage'].isnull().sum())

# Check for how many rows the column hasImage would be False
print(df['hasImage'].value_counts())
from matplotlib import pyplot as plt
df['6_way_label'].plot(kind='hist', bins=20, title='6_way_label')

from urllib import request

# Replace NaN values with empty strings
df = df.replace(np.nan, '', regex=True)
df.fillna('', inplace=True)

print("After split len(df):", len(df))
print("After split len(df_backup):", len(df_backup))
print("Sample ratio kept:", len(df) / (len(df) + len(df_backup)))


### Download, Verify, and Cache the Images

This cell performs the main image preparation loop. It skips rows without images, downloads missing files, verifies integrity immediately after download, and removes corrupted files on failure.

In [ ]:
import os
import time
from urllib import request
from PIL import Image
import numpy as np
import pandas as pd

os.makedirs(IMAGEDIR, exist_ok=True)

df = df.replace(np.nan, '', regex=True).fillna('')

valid_rows = []

stats = {
    "processed": 0,
    "eligible": 0,
    "cached": 0,
    "downloaded": 0,
    "verified": 0,
    "skipped_no_image": 0,
    "failed": 0
}

LOG_EVERY = 100
start_time = time.time()
total_rows = len(df)

print(f"Starting image preparation for {total_rows} rows...")
print(f"Image dir: {IMAGEDIR}")
print(f"Clean CSV path: {CLEANDFPATH}")

for idx, (_, row) in enumerate(df.iterrows(), start=1):
    stats["processed"] += 1

    if row["hasImage"] != True or row["image_url"] in ["", "nan"]:
        stats["skipped_no_image"] += 1
        if idx % LOG_EVERY == 0:
            elapsed = time.time() - start_time
            print(
                f"[{idx}/{total_rows}] "
                f"cached={stats['cached']} downloaded={stats['downloaded']} "
                f"verified={stats['verified']} failed={stats['failed']} "
                f"skipped={stats['skipped_no_image']} elapsed={elapsed:.1f}s"
            )
        continue

    stats["eligible"] += 1

    img_id = str(row["id"])
    image_url = row["image_url"]
    path = os.path.join(IMAGEDIR, f"{img_id}.jpg")

    try:
        if not os.path.exists(path):
            with request.urlopen(image_url, timeout=20) as response:
                image_bytes = response.read()
            with open(path, "wb") as f:
                f.write(image_bytes)
            stats["downloaded"] += 1
            action = "downloaded"
        else:
            stats["cached"] += 1
            action = "cached"

        with Image.open(path) as img:
            img.verify()

        stats["verified"] += 1
        valid_rows.append(row.to_dict())

        if idx % LOG_EVERY == 0:
            elapsed = time.time() - start_time
            print(
                f"[{idx}/{total_rows}] last={img_id} ({action}) "
                f"cached={stats['cached']} downloaded={stats['downloaded']} "
                f"verified={stats['verified']} failed={stats['failed']} "
                f"skipped={stats['skipped_no_image']} elapsed={elapsed:.1f}s"
            )

    except Exception as e:
        stats["failed"] += 1
        print(f"Failed for id={img_id}: {e}")

        if os.path.exists(path):
            os.remove(path)

df = pd.DataFrame(valid_rows).reset_index(drop=True)
df.to_csv(CLEANDFPATH, index=False)

elapsed = time.time() - start_time
print("\nImages ready. Clean df saved.")
print(f"Final rows kept: {len(df)}")
print(f"Stats: {stats}")
print(f"Total time: {elapsed:.1f}s")

### Reload the Saved Clean Dataframe

This cell reloads the cleaned CSV produced by the download stage so the next checks operate on the latest saved version.

In [ ]:
df = pd.read_csv(
    '/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv',
    sep=','
)
df.head(10)

### Inspect the Available Columns

This cell prints the dataframe columns as a quick schema check after the cleaning stage.

In [ ]:
print(df.columns)

### Visually Inspect a Few Downloaded Images

This cell opens a few sample images from disk and displays them to confirm that the download and decode steps produced usable RGB files.

In [ ]:
# Plotting images to test download
for i in range(5):
    path = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/" + df["id"][i] + ".jpg"

    im= np.array(Image.open(path))

    print(im.shape)
    ax= plt.subplot(121)
    ax.imshow(im)

    plt.show()

### Inspect the RGB Channels of a Sample Image

This cell splits one sample image into its red, green, and blue channels so the decoded image can be inspected more closely.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Load the RGBA image
image_path =  "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/" + df["id"][0] + ".jpg"
image = Image.open(image_path).convert("RGB")

# Split the image into individual channels
r, g, b = image.split()

# Plot each channel separately
plt.figure(figsize=(10, 5))

plt.subplot(1, 4, 1)
plt.imshow(r)
plt.title('Red Channel')

plt.subplot(1, 4, 2)
plt.imshow(g)
plt.title('Green Channel')

plt.subplot(1, 4, 3)
plt.imshow(b)
plt.title('Blue Channel')

#plt.subplot(1, 4, 4)
#plt.imshow(a)
#plt.title('Alpha Channel')

plt.tight_layout()
plt.show()

### Run an Initial Corruption Check

This cell performs an initial pass that attempts to open every cached image and removes rows whose files are corrupted.

In [ ]:
def validate_images(directory):
    corrupted_files = []

    # Walk through directory and sub-directories
    for index, row in df.iterrows():
      image_path = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/" + row["id"] + ".jpg"
      try:
          with Image.open(image_path) as img:
              img.verify()
      except Exception as e:
          corrupted_files.append(image_path)
          print(f"Error with {image_path}: {e}")
          df.drop(index = index, axis = 0, inplace = True)

    return corrupted_files

# Example usage:
directory = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images/"
corrupted_images = validate_images(directory)
if corrupted_images:
    print(f"Found {len(corrupted_images)} corrupted images.")
else:
    print("All images are valid!")
df.reset_index(drop=True, inplace=True)

### Run the Refined Validation Pass

This cell defines a more structured validation routine that checks for missing files, verifies image integrity, removes broken cache files, and rebuilds a validated dataframe.

In [ ]:
from PIL import Image
import os
import pandas as pd

def validate_images(frame, imagedir):
    valid_rows = []
    corrupted_files = []

    total_rows = len(frame)
    LOG_EVERY = 100

    print(f"Starting validation for {total_rows} rows...")
    print(f"Image dir: {imagedir}")
    print(f"Clean CSV path: {CLEANDFPATH}")

    for idx, (_, row) in enumerate(frame.iterrows(), start=1):
        img_id = str(row["id"])
        path = os.path.join(imagedir, f"{img_id}.jpg")

        try:
            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing file: {path}")

            with Image.open(path) as img:
                img.verify()

            valid_rows.append(row.to_dict())

            if idx % LOG_EVERY == 0:
                print(f"[{idx}/{total_rows}] validated last={img_id}")

        except Exception as e:
            corrupted_files.append(path)
            print(f"Error with {path}: {e}")

            if os.path.exists(path):
                os.remove(path)

    valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)
    return valid_df, corrupted_files


df, corrupted_images = validate_images(df, IMAGEDIR)
df.to_csv(CLEANDFPATH, index=False)

if corrupted_images:
    print(f"Found {len(corrupted_images)} corrupted/missing images.")
else:
    print("All images are valid!")

print(f"Saved validated dataframe to {CLEANDFPATH}")
print(f"Rows after validation: {len(df)}")

### Resize All Valid Images

This cell resizes each valid cached image to `(256, 256)` and records any files that are still missing or fail during resizing.

In [ ]:
import os
import pandas as pd
from PIL import Image
from torchvision.transforms import v2

# Reload the latest validated CSV
df = pd.read_csv(CLEANDFPATH, sep=",")

new_size = (256, 256)
resize_transform = v2.Resize(new_size)

missing_ids = []
failed_ids = []
resized_count = 0

print(f"Starting resize for {len(df)} rows...")
print(f"Image dir: {IMAGEDIR}")
print(f"Clean CSV path: {CLEANDFPATH}")

for idx, row in df.iterrows():
    img_id = str(row["id"])
    image_path = os.path.join(IMAGEDIR, f"{img_id}.jpg")

    if not os.path.exists(image_path):
        missing_ids.append(img_id)
        print(f"Missing file for id={img_id}")
        continue

    try:
        image = Image.open(image_path).convert("RGB")
        resized_image = resize_transform(image)
        resized_image.save(image_path)
        resized_count += 1

        if (idx + 1) % 100 == 0:
            print(f"[{idx + 1}/{len(df)}] resized last={img_id}")

    except Exception as e:
        failed_ids.append(img_id)
        print(f"Resize failed for id={img_id}: {e}")


print(f"Resized successfully: {resized_count}")
print(f"Missing files: {len(missing_ids)}")
print(f"Failed resize: {len(failed_ids)}")
# Remove rows that still do not have usable image files


### Remove Rows with Unusable Images After Resizing

This cell drops any rows whose images are still missing or failed during resizing, then writes the updated dataframe back to disk.

In [ ]:
bad_ids = set(missing_ids + failed_ids)
if bad_ids:
    df = df[~df["id"].astype(str).isin(bad_ids)].reset_index(drop=True)
    df.to_csv(CLEANDFPATH, index=False)

print("\nResize finished.")

print(f"Final rows kept: {len(df)}")
print(f"Updated clean df saved to: {CLEANDFPATH}")

### Verify a Sample Image on Disk

This cell reloads the cleaned dataframe, opens one sample image, and prints its size and mode as a final spot check.

In [ ]:
import pandas as pd
from PIL import Image
import os

df = pd.read_csv(CLEANDFPATH, sep=",")
print(df.shape)

sample_id = str(df.iloc[0]["id"])
sample_path = os.path.join(IMAGEDIR, f"{sample_id}.jpg")

with Image.open(sample_path) as img:
    print("Sample image size:", img.size)
    print("Sample image mode:", img.mode)

### Confirm the Final Clean CSV

This cell checks that the cleaned CSV exists on disk, prints its modification time, and previews the first few rows.

In [ ]:
import os
import pandas as pd
from datetime import datetime

print("CLEANDFPATH:", CLEANDFPATH)
print("Exists:", os.path.exists(CLEANDFPATH))
print("Modified:", datetime.fromtimestamp(os.path.getmtime(CLEANDFPATH)))

check_df = pd.read_csv(CLEANDFPATH, sep=",")
print("Shape on disk:", check_df.shape)
print(check_df.head(2))

### Persist the Cleaned Dataframe

This cell writes the current dataframe to the clean CSV path and reloads it immediately to confirm the saved shape.

In [ ]:
df.to_csv(CLEANDFPATH, index=False)
print(pd.read_csv(CLEANDFPATH, sep=",").shape)

### Create the Final Dataset Snapshot

This cell copies the cleaned CSV to the final snapshot path and verifies the exported file.

In [ ]:
import shutil
import pandas as pd

CLEANDFPATH = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv"
FINAL_PATH  = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df_final.csv"

shutil.copy2(CLEANDFPATH, FINAL_PATH)

check = pd.read_csv(FINAL_PATH, sep=",")
print("Final shape:", check.shape)   # should print (17294, 13)
print("Columns:", check.columns.tolist())